In [ ]:
import xarray as xr

import eradiate
from eradiate.data.convert import aer_v1_to_aer_core_v2

eradiate.set_mode("mono")

In [ ]:
ds = eradiate.fresolver.load_dataset("aerosol/govaerts_2021-desert-extrapolated.nc")
# ds["phase"] = ds["phase"].assign_attrs(units="1 / sr")
display(ds)
ds = aer_v1_to_aer_core_v2(ds)
display(ds)

In [ ]:
from eradiate.radprops._particles import ParticleProperties
from eradiate.units import unit_registry as ureg

props = ParticleProperties(ds)
print(f"{props.eval_ssa(550. * ureg.nm) = }")
print(f"{props.eval_ext(550. * ureg.nm) = }")
print(f"{props.eval_phase(550. * ureg.nm) = }")

In [ ]:
xr.open_dataset("wc.sol.mie.cdf")

In [ ]:
import drjit as dr
import mitsuba as mi
import numpy as np

mi.set_variant("scalar_mono_double")

mus_a = np.linspace(-1, 1, 11)
values_a = np.linspace(0, 1, 11)
mus_b = np.linspace(-1, 1, 21)
values_b = np.ones_like(mus_b)

mus_init = np.linspace(-1, 1, np.max((len(mus_a), len(mus_b))))
values_init = np.ones_like(mus_init)

# Bug dans tabphase_polarized ? Est-ce que les valeurs des nœuds de la distribution
# sont bien mises à jour quand on fait une update ?
p = mi.load_dict(
    {
        "type": "tabphase_polarized",
        "m11": ",".join(map(str, values_init)),
        "nodes": ",".join(map(str, mus_init)),
    }
)
params = mi.traverse(p)

values_update = np.zeros_like(values_init)
values_update[: len(mus_a)] = values_a
mus_update = np.zeros_like(mus_init)
mus_update[: len(mus_a)] = mus_a
mus_update[len(mus_a) :] = np.linspace(1.00001, 2, len(mus_init) - len(mus_a))

params["nodes"] = mus_update
params["m11"] = values_update
params.update()
print(p)

ctx = dr.zeros(mi.PhaseFunctionContext)
mei = mi.MediumInteraction3f()
mei.wi = (0, 0, 1)
wo = mi.Vector3f(0, 0, 1)
p.eval_pdf(ctx, mei, wo)